# Binned Resolution

Plot the event-level distribution of `nv` versus `logE_pred` for `runs/no_core_cut_2724`.

In [1]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import torch
import uproot
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path('/mnt/mydisk/server/projects/energy_reconstruction')
RUN_DIR = PROJECT_ROOT / 'runs' / 'no_core_cut_2724'
NOTEBOOK_DIR = PROJECT_ROOT / 'notebook'
CONFIG_PATH = RUN_DIR / 'config.json'
TRAIN_STATS_PATH = RUN_DIR / 'dataset_train_stats.json'
VAL_STATS_PATH = RUN_DIR / 'dataset_val_stats.json'
TEST_STATS_PATH = RUN_DIR / 'dataset_test_stats.json'
PREDS_PATH = RUN_DIR / 'fig' / 'preds.npz'
PLOT_PATH = NOTEBOOK_DIR / 'binned_resolution.png'
FILES_PATH = NOTEBOOK_DIR / 'binned_resolution_root_files.txt'
EVENT_TREE = 't_eventout'
BRANCHES = ['fitstat', 'nv', 'vx', 'vy', 'vt', 'vq', 'vqsamp', 'theta', 'dcedge', 'pincness', 'mc_energy', 'mc_dangle', 'mc_xc', 'mc_yc']

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    config = json.load(f)
with open(TRAIN_STATS_PATH, 'r', encoding='utf-8') as f:
    train_stats = json.load(f)
with open(VAL_STATS_PATH, 'r', encoding='utf-8') as f:
    val_stats = json.load(f)
with open(TEST_STATS_PATH, 'r', encoding='utf-8') as f:
    test_stats = json.load(f)

root_dir = Path(config['root_path'])
all_root_files = sorted(str(p) for p in root_dir.iterdir() if p.suffix == '.root' and p.is_file())
root_files = all_root_files[: int(config['n_files'])]
train_files, test_files = train_test_split(root_files, test_size=float(config['test_size']), random_state=int(config['seed']))
train_files, val_files = train_test_split(train_files, test_size=float(config['val_size']), random_state=int(config['seed']))

cuts = {
    'Emin': config.get('Emin'),
    'Emax': config.get('Emax'),
    'pinc_max': config.get('pinc_max'),
    'dcedge_min': config.get('dcedge_min'),
    'dangle_max_rad': np.deg2rad(config['dangle_max_deg']) if config.get('dangle_max_deg') is not None else None,
    'theta_max_rad': np.deg2rad(config['theta_max_deg']) if config.get('theta_max_deg') is not None else None,
    'use_core_box': bool(config.get('use_core_box', False)),
    'core_box': tuple(config.get('core_box', [-130.0, 130.0, -110.0, 110.0])),
    'vqsamp_ratio_min': config.get('vqsamp_ratio_min'),
    'require_fitstat0': config.get('require_fitstat0') if config.get('require_fitstat0') is not None else True,
    'fitstat_equals': config.get('fitstat_equals') if config.get('fitstat_equals') is not None else 0,
}


def build_mask(arrays):
    mask = np.ones(len(arrays['nv']), dtype=bool)
    if cuts['Emin'] is not None:
        mask &= arrays['mc_energy'] > float(cuts['Emin'])
    if cuts['Emax'] is not None:
        mask &= arrays['mc_energy'] < float(cuts['Emax'])
    if cuts['pinc_max'] is not None:
        mask &= arrays['pincness'] < float(cuts['pinc_max'])
    if cuts['dcedge_min'] is not None:
        mask &= arrays['dcedge'] > float(cuts['dcedge_min'])
    if cuts['dangle_max_rad'] is not None:
        mask &= arrays['mc_dangle'] < float(cuts['dangle_max_rad'])
    if cuts['theta_max_rad'] is not None:
        mask &= arrays['theta'] < float(cuts['theta_max_rad'])
    if cuts['require_fitstat0']:
        mask &= arrays['fitstat'] == int(cuts['fitstat_equals'])
    if cuts['use_core_box']:
        xmin, xmax, ymin, ymax = cuts['core_box']
        mask &= (
            (arrays['mc_xc'] >= xmin)
            & (arrays['mc_xc'] <= xmax)
            & (arrays['mc_yc'] >= ymin)
            & (arrays['mc_yc'] <= ymax)
        )
    if cuts['vqsamp_ratio_min'] is not None:
        ratio = np.array([
            (np.count_nonzero(v > 0) / len(v)) if len(v) else 0.0
            for v in arrays['vqsamp']
        ], dtype=np.float32)
        mask &= ratio >= float(cuts['vqsamp_ratio_min'])
    hit_counts = np.asarray([len(v) for v in arrays['vx']], dtype=np.int64)
    mask &= hit_counts > 0
    return mask


torch.manual_seed(int(config['seed']))
_ = torch.randperm(int(train_stats['events']['n_kept']))
_ = torch.randperm(int(val_stats['events']['n_kept']))

nv_chunks = []
file_map = {file_path: EVENT_TREE for file_path in test_files}
for arrays in uproot.iterate(file_map, BRANCHES, library='np', step_size='128 MB'):
    mask = build_mask(arrays)
    nv_chunks.append(np.asarray(arrays['nv'][mask]).reshape(-1))

nv = np.concatenate(nv_chunks, axis=0)
nv = nv[torch.randperm(len(nv)).numpy()]
logE_pred = np.asarray(np.load(PREDS_PATH)['logE_pred'], dtype=np.float32).reshape(-1)

n = min(len(nv), len(logE_pred))
if len(nv) != len(logE_pred):
    print(f'warning: nv length {len(nv)} != logE_pred length {len(logE_pred)}; plotting first {n} entries')
nv = nv[:n]
logE_pred = logE_pred[:n]

FILES_PATH.write_text('\n'.join(test_files) + '\n', encoding='utf-8')

fig, ax = plt.subplots(figsize=(8.5, 6.5))
hb = ax.hexbin(nv, logE_pred, gridsize=100, bins='log', mincnt=1, cmap='viridis')
ax.set_xlabel('nv (nhit)')
ax.set_ylabel('logE_pred')
ax.set_title('no_core_cut_2724: nv vs logE_pred')
cbar = fig.colorbar(hb, ax=ax)
cbar.set_label('log10(count)')
fig.tight_layout()
fig.savefig(PLOT_PATH, dpi=300, bbox_inches='tight')
plt.show()

print(f'plot saved: {PLOT_PATH}')
print(f'root file list saved: {FILES_PATH}')
print(f'logE_pred shape: {logE_pred.shape}')
print(f'nv shape: {nv.shape}')
print(f'test files: {len(test_files)}')
print('first 5 test files:')
for file_path in test_files[:5]:
    print('  ', file_path)


SyntaxError: unterminated string literal (detected at line 103) (1586127685.py, line 103)